# Project analysis - IT jobs situation over the years

In [43]:
import requests
import json
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import dendrogram
from scipy.spatial.distance import pdist
import country_converter as coco

In [44]:
URL_API = 'https://upy-homeworks.xpert-ia.com.mx/visualization-tools/'

Getting a sample (10 records) of the data available in the database using the API.

In [45]:
"""response_sample = requests.get(url=f'{URL_API}api/postgres', params={
    'table': 'jobs_proyect_u2',
    'limit': 10
})
payload_sample = response_sample.json()
payload_sample"""

"response_sample = requests.get(url=f'{URL_API}api/postgres', params={\n    'table': 'jobs_proyect_u2',\n    'limit': 10\n})\npayload_sample = response_sample.json()\npayload_sample"

Getting all the records available in the database.

In [46]:
"""response = requests.get(url=f'{URL_API}api/postgres', params={
    'table': 'jobs_proyect_u2'
})
payload = response.json()"""

"response = requests.get(url=f'{URL_API}api/postgres', params={\n    'table': 'jobs_proyect_u2'\n})\npayload = response.json()"

### Transforming the JSON response into a dataframe

In [47]:
"""base_df = pd.json_normalize(payload['data']).drop(columns=['id']).sort_values(by='work_year')
base_df"""

"base_df = pd.json_normalize(payload['data']).drop(columns=['id']).sort_values(by='work_year')\nbase_df"

Saving the data into a file to avoid calling the API multiple times.

In [48]:
"""base_df.to_csv('./data/projectU2.csv', index= False)"""

"base_df.to_csv('./data/projectU2.csv', index= False)"

# Getting the data from CSV

Loading the base dataset

In [49]:
base_df = pd.read_csv('./data/projectU2.csv').drop(columns=['created_at', 'updated_at'])
base_df

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,category,subcategory,department,employee_latitude,employee_longitude,employee_country_name,company_latitude,company_longitude,company_country_name
0,2022,SE,FT,Cybersecurity,55000,GBP,67723,GB,50,GB,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,55.378050,-3.435973,Reino Unido,55.378050,-3.435973,Reino Unido
1,2022,MI,FT,Cloud Security Engineer,41000,EUR,43077,IT,100,IT,L,Security Engineering & Architecture,Cloud & Network Security,Cybersecurity,41.871940,12.567380,Italia,41.871940,12.567380,Italia
2,2022,EN,FT,IT Security Analyst,52000,USD,52000,US,0,US,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
3,2023,SE,FT,Principal Security Architect,210000,USD,210000,US,100,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
4,2023,SE,FT,Principal Security Architect,200000,USD,200000,US,100,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57245,2025,SE,FT,Security Engineer,200000,USD,200000,US,0,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57246,2025,SE,FT,Security Engineer,350000,USD,350000,US,0,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57247,2025,SE,FT,Security Analyst,153900,USD,153900,US,0,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57248,2025,SE,FT,Security Analyst,266800,USD,266800,US,0,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos


In [50]:
columns = list(base_df.columns)
print(columns)

['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size', 'category', 'subcategory', 'department', 'employee_latitude', 'employee_longitude', 'employee_country_name', 'company_latitude', 'company_longitude', 'company_country_name']


In [51]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57250 entries, 0 to 57249
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   work_year              57250 non-null  int64  
 1   experience_level       57250 non-null  object 
 2   employment_type        57250 non-null  object 
 3   job_title              57250 non-null  object 
 4   salary                 57250 non-null  int64  
 5   salary_currency        57250 non-null  object 
 6   salary_in_usd          57250 non-null  int64  
 7   employee_residence     57250 non-null  object 
 8   remote_ratio           57250 non-null  int64  
 9   company_location       57250 non-null  object 
 10  company_size           57250 non-null  object 
 11  category               57250 non-null  object 
 12  subcategory            57250 non-null  object 
 13  department             57250 non-null  object 
 14  employee_latitude      57250 non-null  float64
 15  em

In [52]:
base_df.isnull().sum()

work_year                0
experience_level         0
employment_type          0
job_title                0
salary                   0
salary_currency          0
salary_in_usd            0
employee_residence       0
remote_ratio             0
company_location         0
company_size             0
category                 0
subcategory              0
department               0
employee_latitude        0
employee_longitude       0
employee_country_name    0
company_latitude         0
company_longitude        0
company_country_name     0
dtype: int64

In [53]:
base_df = base_df.replace(to_replace={'remote_ratio': {0: 'Remote', 50: 'Hybrid', 100: 'On-site'}})
base_df

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,category,subcategory,department,employee_latitude,employee_longitude,employee_country_name,company_latitude,company_longitude,company_country_name
0,2022,SE,FT,Cybersecurity,55000,GBP,67723,GB,Hybrid,GB,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,55.378050,-3.435973,Reino Unido,55.378050,-3.435973,Reino Unido
1,2022,MI,FT,Cloud Security Engineer,41000,EUR,43077,IT,On-site,IT,L,Security Engineering & Architecture,Cloud & Network Security,Cybersecurity,41.871940,12.567380,Italia,41.871940,12.567380,Italia
2,2022,EN,FT,IT Security Analyst,52000,USD,52000,US,Remote,US,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
3,2023,SE,FT,Principal Security Architect,210000,USD,210000,US,On-site,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
4,2023,SE,FT,Principal Security Architect,200000,USD,200000,US,On-site,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57245,2025,SE,FT,Security Engineer,200000,USD,200000,US,Remote,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57246,2025,SE,FT,Security Engineer,350000,USD,350000,US,Remote,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57247,2025,SE,FT,Security Analyst,153900,USD,153900,US,Remote,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos
57248,2025,SE,FT,Security Analyst,266800,USD,266800,US,Remote,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,37.090240,-95.712890,Estados Unidos,37.090240,-95.712890,Estados Unidos


# Time series analysis

## _**Temporal analysis**_

### **PROVISIONAL IDEA TO ENHANCE THE TEMPORAL ANALYSIS IF THE BASE DATASET IS NOT ENOUGH**

For an enhanced temporal analysis, we will add another dataset in order to make, for example, ACF and CCF. The new dataset is ``jobs_ai_skills.csv``, which contains the increase of jobs (in certain countries from 2020 to 2025) that list AI skills as requirements.

In [54]:
support_df = pd.read_csv('./data/jobs_ai_skills.csv').rename(columns={'Share of artificial intelligence jobs among all job postings': 'Jobs_ai_skills(%)'})
support_df = support_df[support_df['Year'] >= 2020]
support_df

,Entity,Code,Year,Jobs_ai_skills(%)
6,Australia,AUS,2020,0.82
7,Australia,AUS,2021,1.17
8,Australia,AUS,2022,1.11
9,Australia,AUS,2023,0.91
10,Australia,AUS,2024,1.14
...,...,...,...,...
129,United States,USA,2020,1.37
130,United States,USA,2021,1.49
131,United States,USA,2022,1.69
132,United States,USA,2023,1.39


In [55]:
"""base_df = base_df.merge(right=[support_df], left_on='')"""

"base_df = base_df.merge(right=[support_df], left_on='')"

## _**Spatial analysis**_

In [56]:
"""geo_df = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
geo_df"""

'geo_df = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")\ngeo_df'

In [57]:
"""geo_df.plot()"""

'geo_df.plot()'

## _**Spatiotemporal analysis**_

# Hierarchical analysis

In [58]:
base_df['department'].unique()

array(['Cybersecurity', 'Executive & Leadership',
       'Engineering & Development', 'Sales & Consulting',
       'Information Technology (IT)', 'Legal & Compliance',
       'Finance & Accounting', 'Operations', 'Data & Analytics',
       'Human Resources (HR)', 'Product Management'], dtype=object)

['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size', 'category', 'subcategory', 'department', 'employee_latitude', 'employee_longitude', 'employee_country_name', 'company_latitude', 'company_longitude', 'company_country_name']

In [59]:
jobs_df = base_df.drop(columns=['work_year', 'employee_latitude', 'employee_longitude', 'company_latitude', 'company_longitude', 'salary', 'salary_currency'])
jobs_df

,experience_level,employment_type,job_title,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,category,subcategory,department,employee_country_name,company_country_name
0,SE,FT,Cybersecurity,67723,GB,Hybrid,GB,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,Reino Unido,Reino Unido
1,MI,FT,Cloud Security Engineer,43077,IT,On-site,IT,L,Security Engineering & Architecture,Cloud & Network Security,Cybersecurity,Italia,Italia
2,EN,FT,IT Security Analyst,52000,US,Remote,US,L,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,Estados Unidos,Estados Unidos
3,SE,FT,Principal Security Architect,210000,US,On-site,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,Estados Unidos,Estados Unidos
4,SE,FT,Principal Security Architect,200000,US,On-site,US,L,Security Engineering & Architecture,Security Architecture,Executive & Leadership,Estados Unidos,Estados Unidos
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57245,SE,FT,Security Engineer,200000,US,Remote,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,Estados Unidos,Estados Unidos
57246,SE,FT,Security Engineer,350000,US,Remote,US,M,Security Engineering & Architecture,Security Engineering,Cybersecurity,Estados Unidos,Estados Unidos
57247,SE,FT,Security Analyst,153900,US,Remote,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,Estados Unidos,Estados Unidos
57248,SE,FT,Security Analyst,266800,US,Remote,US,M,Security Engineering & Architecture,General Security Analysis & Administration,Cybersecurity,Estados Unidos,Estados Unidos


In [60]:
departments_list = list(jobs_df['department'].unique())
print(departments_list)

['Cybersecurity', 'Executive & Leadership', 'Engineering & Development', 'Sales & Consulting', 'Information Technology (IT)', 'Legal & Compliance', 'Finance & Accounting', 'Operations', 'Data & Analytics', 'Human Resources (HR)', 'Product Management']


In [61]:
employees_number = []
incomes_per_sub = []
subcategories = []
categories = []
departments = []

for department in departments_list:
    department_df = jobs_df[jobs_df['department'] == department]
    categories_list = department_df['category'].unique()
    for category in categories_list:
        category_df = department_df[department_df['category'] == category]
        subcategories_list = category_df['subcategory'].unique()
        for subcategory in subcategories_list:
            subcategory_df = category_df[category_df['subcategory'] == subcategory]
            income = subcategory_df['salary_in_usd'].sum()
            employees_number.append(subcategory_df['subcategory'].count())
            incomes_per_sub.append(income)
            subcategories.append(subcategory)
            categories.append(category)
            departments.append(department)
    
jobs_df2 = pd.DataFrame({
    'department': departments,
    'category': categories,
    'subcategory': subcategories,
    'employees': employees_number,
    'income': incomes_per_sub
})

jobs_df2


,department,category,subcategory,employees,income
0,Cybersecurity,Security Engineering & Architecture,General Security Analysis & Administration,7728,919019698
1,Cybersecurity,Security Engineering & Architecture,Cloud & Network Security,67,12522513
2,Cybersecurity,Security Engineering & Architecture,Security Engineering,12990,2070678783
3,Cybersecurity,Security Engineering & Architecture,Identity & Access Management (IAM),270,36327475
4,Cybersecurity,Threat Intelligence & Forensics,Threat Intelligence,991,128704814
...,...,...,...,...,...
74,Data & Analytics,Data & AI Security,Data Engineering & Architecture,102,12757442
75,Data & Analytics,Data & AI Security,Data Science & Analytics,184,26741791
76,Data & Analytics,Data & AI Security,AI/ML & Data Security,30,4904294
77,Human Resources (HR),Training & Education,Technical Training & Education,8,828299


In [ ]:
treemap_chart = px.treemap(
    data_frame= jobs_df2,
    values='income',
    path=['department', 'category', 'subcategory'],
    title='Departments presence in IT Jobs',
    color='income',  # Color by subcategory income to show the incomes per department
    color_continuous_scale="peach"
)

treemap_chart.update_traces(textinfo="label+value", textfont_size=11)
treemap_chart.update_layout(height=700, width=1000)
treemap_chart.show()

In [63]:
sunburst_chart = px.sunburst(
    data_frame=jobs_df2,
    title='Departments presence in IT',
    height=600,
    path=['department', 'category', 'subcategory'],
    values='employees',
    color='employees',
    color_continuous_scale=px.colors.sequential.Viridis
)

# Customize layout for better display
sunburst_chart.update_layout(
    font=dict(size=12),
    title_x=0.5, # Center the title
)

# Showing the organizational structure
sunburst_chart.show()